# Week 3 Day 4 Lab 2 (free route) - UnderCover on Render

## Running the agent beside the app

On the AWS route, Lab 2 deploys the agent on AgentCore Runtime, where each dispatch runs in a session of its own, and points the app on Lambda at it. On the free route we run the agent as a second web service on Render, beside your app from Day 2, with the context layer inside the same service.

AgentCore Runtime, the Gateway and the Memory are AWS services, so for those you can watch me. The rest of today's build works the same way: the agent is the same AgentCore app, the app starts its work and keeps one run at a time, and a senior handler can send a claim back to the agent with a note.

This notebook has no code to run. It gives you the steps, with links to Render's own documentation.

## 5 steps:

STEP 1: Update the app on Render

STEP 2: Create the agent's web service

STEP 3: Point the app at the agent

STEP 4: Send a claim back

STEP 5: Compare with the AWS route

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This is the free version of today's second lab. The main lab, <a href="week3day4_lab2.ipynb">week3day4_lab2.ipynb</a>, runs on AWS. You need the Render service from Day 2, and today's build of the product, yours or <code>week3day4-end</code>, pushed to GitHub as on Day 2.<br/><br/>I do this week on AWS, so I'm not recording the Render steps. Render's website changes from time to time, so if a screen looks different from what's described here, follow the linked page in Render's documentation.<br/><br/>Step 4 works one claim and then works it again, which uses part of your 50 free requests for the day. A dispatch works the whole queue, which needs more requests than a day's allowance, so on the free route we work one claim.
            </span>
        </td>
    </tr>
</table>

## Step 1: Update the app on Render

Today's build changes the app: it keeps one run at a time itself, and a senior handler can send a claim back. So the app on Render needs today's code.

- **To deploy my build,** open your web service in the [Render Dashboard](https://dashboard.render.com), go to **Settings**, change the **Branch** to `week3day4-end`, and choose **Save changes**. Render starts the deploy straight away.
- **To deploy your own build,** push your branch to your own GitHub repository, as on Day 2, and choose **Manual Deploy** and **Deploy latest commit**.

The build and start commands and the environment variables stay as they were on Day 2. Step 3 adds one more.

Render's documentation: [Deploys](https://render.com/docs/deploys).

## Step 2: Create the agent's web service

In the [Render Dashboard](https://dashboard.render.com), choose **New**, then **Web Service**, and pick the same repository and branch as your app. Then fill in the form:

| Field | Value |
|---|---|
| Name | `undercover-agent` |
| Language | Python 3 |
| Region | The same as your app |
| Build Command | `uv sync --frozen --only-group runtime --only-group context` |
| Start Command | `uv run --no-sync python -m context & uv run --no-sync uvicorn agent.entrypoint:app --host 0.0.0.0 --port $PORT` |
| Compute | $0 / month |

As on Day 2, replace any Build Command Render fills in, and under **Compute** choose **$0 / month**, since Render selects $7 / month again.

Under **Advanced**, set the **Health Check Path** to `/ping`.

What the commands do:

- `--only-group runtime` installs the packages the agent needs on a server, which the AWS route uses on Runtime, and `--only-group context` adds the context layer's.
- The start command runs the context layer in the background, at `http://127.0.0.1:8006/mcp` inside the service, where the agent looks for it by default, and the agent's AgentCore app on Render's port, as `npm run dev` serves it on your laptop. Only the agent is on the internet.

Under **Environment Variables**, add these lines from your `.env` file, and one more:

| Key | Value |
|---|---|
| `OPENROUTER_API_KEY` | From your `.env` |
| `UNDERCOVER_API_KEY` | From your `.env`, the same key as the app's |
| `DATABASE_URL` | The same session pooler address as in your `.env` |
| `UNDERCOVER_API` | Your app's address on Render, such as `https://undercover.onrender.com` |

Away from AgentCore Runtime, the agent refuses any request without the agent's key, so only your app can start its work. Choose **Deploy web service**, and when the deploy is live, `https://<your-agent>.onrender.com/ping` answers with `"status": "Healthy"`.

The service's logs on Render show an "Authentication error" from Langfuse, saying its client is disabled. That's expected: the service has no Langfuse keys, so the agent runs without tracing.

Render's documentation: [Web services](https://render.com/docs/web-services), [Environment variables and secrets](https://render.com/docs/configure-environment-variables), [Health checks](https://render.com/docs/health-checks).

## Step 3: Point the app at the agent

Open your app's web service, and under **Environment**, add:

| Key | Value |
|---|---|
| `UNDERCOVER_AGENT` | Your agent's address on Render, such as `https://undercover-agent.onrender.com` |

Choose **Edit**, then **Add variable**, and save with **Save and deploy**. It's under the arrow beside **Save, rebuild, and deploy**, which also works but rebuilds the app first. The app now sends the agent's work to the agent on Render, with the agent's key, as it does on your laptop.

A free service spins down after 15 minutes without a request, and the first request after that takes about a minute. The app waits only 10 seconds for the agent, so before you start any work, open the agent's `/ping` address in your browser and wait until it answers.

Render's documentation: [Free instances](https://render.com/docs/free).

## Step 4: Send a claim back

The three approvals over the limit in the queue are CLM-1003, CLM-1008 and CLM-1011. In a terminal in the `undercover` folder, reseed and start everything:

```
npm run seed
npm run dev
```

In a second terminal, work one of them with the agent on your laptop:

```
uv run -m agent CLM-1011
```

If the agent approves it, the claim waits for sign-off, with the claim graph paused and its place saved in Supabase. If it declines or refers the claim instead, try one of the other two. Then stop `npm run dev` with Ctrl+C. The rest happens on Render, and Supabase's free plan allows only 15 database connections at once.

Open your app on Render and sign in as Dana Whitfield. On the claim's page, read the decision and send it back with a note saying what to look at again. A free model's mistakes vary from run to run. On CLM-1011, check that the decision takes off the $350 Meridian Air has already paid the customer, and that it counts a pair or a set, such as the suit or the camera with its lens, as one item. If both look right, send it back with a note asking the agent to check one of them.

The app on Render asks the agent on Render to resume the claim. The agent loads the claim's thread from Supabase, where your laptop left it, reads the note and decides again. Refresh the claim's page to watch its steps.

## Step 5: Compare with the AWS route

On the AWS route, the same send back starts a new session on AgentCore Runtime, which loads the claim's thread from Supabase in the same way. The agent is the same AgentCore app on both. The difference is in how it runs: on Render it's one process that stays up between requests and sleeps after 15 minutes without one, while Runtime starts a session for each piece of work, in a microVM of its own, and ends it when the work is done. Watch Part 5 of the main lab for a dispatch of 25 claim agents in one session.

There's nothing to tear down. Both free web services cost nothing, so keep them for Day 5. You can delete them from the Render Dashboard whenever you no longer need them.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Challenge: the app keeps one run at a time in the database, and hears from the agent every minute while it works. Stop the agent from the Render Dashboard partway through a dispatch or an evaluation, start it again, and see what the app says when you start new work, before and after five minutes.<br/><br/>I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>

In [1]:
# Local verification of the Render (free route) deployment.
import os
import time
from datetime import datetime, timezone

import requests
from dotenv import load_dotenv

load_dotenv("../.env", override=True)
APP = os.getenv("UNDERCOVER_API", "https://undercover-ap.onrender.com").rstrip("/")
print("Checked at (UTC):", datetime.now(timezone.utc).isoformat(timespec="seconds"))
print("App:", APP)


def check(path):
    started = time.time()
    try:
        response = requests.get(APP + path, timeout=120)
        body = response.text.strip().replace("\n", " ")[:160]
        return f"{path}: HTTP {response.status_code} in {time.time() - started:.1f}s - {body}"
    except Exception as exc:
        return f"{path}: {type(exc).__name__}: {str(exc)[:160]}"


for path in ["/api/health", "/sign-in", "/api/claims/CLM-1901"]:
    print(check(path))

for name in ["UNDERCOVER_AGENT", "UNDERCOVER_COPILOT"]:
    print(f"{name} setting:", os.getenv(name, "not set locally (set in Render dashboard)"))


Checked at (UTC): 2026-11-27T05:31:47+00:00
App: https://undercover-ap.onrender.com


/api/health: HTTP 200 in 0.5s - {"status":"ok"}


/sign-in: HTTP 200 in 0.6s - <!doctype html> <html lang="en">   <head>     <meta charset="UTF-8" />     <meta name="viewport" content="width=device-width, initial-scale=1.0" />     <title>U


/api/claims/CLM-1901: HTTP 401 in 0.6s - {"detail":"Sign in, or send the agent's API key"}
UNDERCOVER_AGENT setting: not set locally (set in Render dashboard)
UNDERCOVER_COPILOT setting: not set locally (set in Render dashboard)


## Solution - assignment challenge

### What happens when the agent on Render is stopped mid-run and started again

This is what the app's own code and `docs/TECH_STACK.md` imply; I have not run the stop/restart experiment in this session.

**What the app knows while work is going.** The app is a web service with no in-memory record of the agent. One row in the `locks` table stands for the agent's work, and a start takes that row with a conditional update as the transaction's first write. A dispatch or evaluation row is `running` with a `heard_at` timestamp; the agent's `beating()` context manager calls `record_run_heartbeat` every 60 seconds from a thread of its own, so `heard_at` stays current. The app treats work it has not heard from for five minutes as stopped:

```python
QUIET = timedelta(minutes=5)
GONE_QUIET = "The agent stopped answering"
GOING = {
    RunKind.DISPATCH: "A dispatch is running",
    RunKind.EVALUATION: "An evaluation is running",
}
```

Each claim's LangGraph state is saved in Supabase after every step with `durability="sync"`, so the app does not need to know how far the agent got.

**Stop the agent, restart it immediately, try new work within five minutes.** Killing the Render service kills the work thread and its heartbeat, but nothing tells the app. The old run is still `running` and its `heard_at` is still recent. On the next start, `clear_the_way()` takes the lock, calls `stop_quiet_work()` (which marks nothing stopped, because the silence is shorter than five minutes), then calls `refuse_while_work_goes_on()` and returns 409: "A dispatch is running" or "An evaluation is running" (or "Round N is running" when it is an evaluation inside an improvement round). The dashboard shows the refusal and the original run still shows as running with no summary. Restarting the agent does **not** resume the old run by itself: the agent gets ready on its first request but deliberately stops no work, and the app only asks the agent to work when a run is started. The half-worked claims stay `in_progress`, with their checkpoints intact. A cold free service also takes about a minute to wake, while the app waits only ten seconds for it, so I would open `/ping` first and wait for "Healthy" before trying again.

**Wait more than five minutes, then start new work.** At that start, `stop_quiet_work()` finds the old run's `heard_at` older than the cutoff and marks it `stopped` with `stopped_because = "The agent stopped answering"`. That commit survives however the rest of the start goes. The dashboard now shows the old dispatch or evaluation as stopped with that reason; `finish_run()` on a stopped run returns it unchanged, with no summary and no grades. The new run is created and offered to the restarted agent, which works it normally.

**What happens to the claims.** For a dispatch, the new dispatcher lists claims in progress, waiting and sent back, oldest submitted first. So the claims that were half done are picked up again. `work_in_run()` carries an in-progress claim on from its saved checkpoint on the same thread and the same attempt; it does not start a new attempt. Steps already recorded stay on the claim, and a repeated decision or payment is treated as already done, so no claim is decided or paid twice. Claims decided before the stop are not reworked. If one claim still fails, the dispatcher retries transient errors and otherwise leaves that claim for the next dispatch; one bad claim does not stop the run. The new dispatch's summary is built from the records, so it counts the carried-over claims once each.

For an evaluation the story is slightly different. Its copies are rows of their own, and a new evaluation creates fresh copies from the labeled set. The interrupted evaluation's copies stay in progress and are never graded; the stopped evaluation does not get a score. The agent does not silently finish the old one after the restart. If the evaluation belonged to an improvement round, the app stops the round as well, because an evaluation in a round is part of the round's work and the round's own heartbeat is what covers it.

**Why the design is like this.** The five-minute quiet period is deliberately much longer than the one-minute heartbeat, so a slow heartbeat or a brief Render restart does not produce a false stop. Keeping the lock in the database, rather than in the agent process, means the app can give a clear answer to a second start even after the process that held the work has gone. Keeping checkpoints in Supabase means restarting any service loses no model work. The practical rule for a handler is: if you stop the agent mid-run, expect a 409 for up to five minutes after you restart it; then start again and look for "Stopped: The agent stopped answering" on the old run. Warm `/ping` before starting, and never delete or reseed the database to clear the lock, because that would also throw away the checkpoints and the claims' history.